# Notebook 3 — SQL Analytics

**Team 8 — Delhi Air Quality Pipeline**
**Member: [Your Name Here]**

## What this notebook does
This is the third and final stage of the pipeline. My job is to:
1. Connect to the SQLite database that Member 2 produced
2. Write SQL queries that answer real questions about Delhi air quality
3. Interpret the results
4. Visualize key findings with charts

## Questions I'm answering
1. How does PM2.5 vary by month? Is it getting better or worse?
2. Which monitoring stations consistently show the worst air quality?
3. What's the worst hour of the day for pollution?
4. How do pollution levels differ by season (winter vs summer vs monsoon)?
5. Does humidity correlate with pollution levels?
6. How many days had genuinely hazardous air (PM2.5 > 250)?


## Step 1: Upload the database from Member 2

In [ ]:
from google.colab import files
uploaded = files.upload()

In [ ]:
import sqlite3
import pandas as pd

conn = sqlite3.connect("delhi_air_quality.db")
print("Connected to database")

## Step 2: Quick check — what's in the database?

In [ ]:
# list the tables
tables = pd.read_sql("SELECT name FROM sqlite_master WHERE type='table'", conn)
print("Tables in the database:")
print(tables)

In [ ]:
# row counts for each table
for table in ["stations", "pollutants", "measurements"]:
    count = pd.read_sql(f"SELECT COUNT(*) AS n FROM {table}", conn).iloc[0, 0]
    print(f"{table}: {count:,} rows")

## Query 1: Monthly PM2.5 trend

How does average PM2.5 vary across the months in our dataset? This tells us
whether pollution is getting worse, better, or seasonal.

In [ ]:
query1 = '''
SELECT
    m.year,
    m.month,
    ROUND(AVG(m.avg_value), 2) AS avg_pm25,
    COUNT(*) AS data_points
FROM measurements m
JOIN pollutants p ON m.pollutant_id = p.pollutant_id
WHERE p.pollutant_name = 'pm25'
GROUP BY m.year, m.month
ORDER BY m.year, m.month
'''
result1 = pd.read_sql(query1, conn)
result1

In [ ]:
# visualize it
import matplotlib.pyplot as plt

result1["period"] = result1["year"].astype(str) + "-" + result1["month"].astype(str).str.zfill(2)

plt.figure(figsize=(12, 4))
plt.plot(result1["period"], result1["avg_pm25"], marker="o")
plt.xticks(rotation=45)
plt.ylabel("Average PM2.5 (µg/m³)")
plt.xlabel("Month")
plt.title("Delhi Monthly Average PM2.5")
plt.axhline(y=35, color="orange", linestyle="--", label="WHO unhealthy threshold (35)")
plt.axhline(y=250, color="red", linestyle="--", label="Hazardous threshold (250)")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

**What I see:** PM2.5 is highest in winter months (November–January) and
much lower in monsoon months (July–September). This matches what's well-known
about Delhi — crop burning, low wind, and temperature inversion in winter trap
pollutants near the ground.

## Query 2: Top 5 most polluted stations

Which monitoring stations show the worst PM2.5 readings on average?

In [ ]:
query2 = '''
SELECT
    s.station_name,
    ROUND(AVG(m.avg_value), 2) AS avg_pm25,
    COUNT(*) AS hourly_readings
FROM measurements m
JOIN stations s ON m.station_id = s.station_id
JOIN pollutants p ON m.pollutant_id = p.pollutant_id
WHERE p.pollutant_name = 'pm25'
GROUP BY s.station_name
ORDER BY avg_pm25 DESC
LIMIT 5
'''
result2 = pd.read_sql(query2, conn)
result2

## Query 3: Worst hour of the day

Is air quality worse in the morning, afternoon, or night?

In [ ]:
query3 = '''
SELECT
    m.hour,
    ROUND(AVG(m.avg_value), 2) AS avg_pm25
FROM measurements m
JOIN pollutants p ON m.pollutant_id = p.pollutant_id
WHERE p.pollutant_name = 'pm25'
GROUP BY m.hour
ORDER BY m.hour
'''
result3 = pd.read_sql(query3, conn)

plt.figure(figsize=(10, 4))
plt.bar(result3["hour"], result3["avg_pm25"], color="steelblue")
plt.xlabel("Hour of day")
plt.ylabel("Average PM2.5 (µg/m³)")
plt.title("PM2.5 by hour of day (Delhi, all stations averaged)")
plt.xticks(range(0, 24))
plt.grid(True, alpha=0.3, axis="y")
plt.tight_layout()
plt.show()

result3

**What I see:** Pollution typically peaks late at night and early morning
(when cold air traps pollutants near the ground), dips in the afternoon (when
the sun heats up the air and mixes it), and rises again in the evening.

## Query 4: Seasonal comparison across pollutants

Compare pollution levels across four seasons for the major pollutants.

In [ ]:
query4 = '''
SELECT
    CASE
        WHEN m.month IN (12, 1, 2) THEN 'Winter'
        WHEN m.month IN (3, 4, 5) THEN 'Summer'
        WHEN m.month IN (6, 7, 8, 9) THEN 'Monsoon'
        ELSE 'Post-monsoon'
    END AS season,
    p.pollutant_name,
    ROUND(AVG(m.avg_value), 2) AS avg_value
FROM measurements m
JOIN pollutants p ON m.pollutant_id = p.pollutant_id
WHERE p.pollutant_name IN ('pm25', 'pm10', 'no2', 'ozone', 'so2')
GROUP BY season, p.pollutant_name
ORDER BY p.pollutant_name, avg_value DESC
'''
result4 = pd.read_sql(query4, conn)

# reshape for a nicer comparison view
pivot = result4.pivot(index="pollutant_name", columns="season", values="avg_value")
pivot[["Winter", "Post-monsoon", "Summer", "Monsoon"]]

## Query 5: Does humidity correlate with pollution?

Group hourly readings into humidity buckets and compare PM2.5 averages.

In [ ]:
query5 = '''
SELECT
    CASE
        WHEN avg_humidity < 30 THEN '1. Dry (<30%)'
        WHEN avg_humidity < 60 THEN '2. Moderate (30-60%)'
        WHEN avg_humidity < 80 THEN '3. Humid (60-80%)'
        ELSE '4. Very humid (>80%)'
    END AS humidity_bucket,
    ROUND(AVG(m.avg_value), 2) AS avg_pm25,
    COUNT(*) AS readings
FROM measurements m
JOIN pollutants p ON m.pollutant_id = p.pollutant_id
WHERE p.pollutant_name = 'pm25' AND avg_humidity IS NOT NULL
GROUP BY humidity_bucket
ORDER BY humidity_bucket
'''
result5 = pd.read_sql(query5, conn)
result5

## Query 6: How many hazardous days?

WHO classifies PM2.5 above 250 µg/m³ as hazardous. Count how many
station-days had genuinely hazardous air quality.

In [ ]:
query6 = '''
SELECT
    m.year,
    m.month,
    COUNT(DISTINCT m.station_id || '-' || m.day) AS hazardous_station_days
FROM measurements m
JOIN pollutants p ON m.pollutant_id = p.pollutant_id
WHERE p.pollutant_name = 'pm25' AND m.avg_value > 250
GROUP BY m.year, m.month
ORDER BY m.year, m.month
'''
result6 = pd.read_sql(query6, conn)
result6

## Findings summary

| Finding | Evidence |
|---|---|
| Winter is by far Delhi's worst pollution season | Query 1 (monthly trend), Query 4 (seasonal) |
| Pollution peaks at night and early morning | Query 3 (hourly pattern) |
| Some stations consistently show worse air than others | Query 2 (top 5) |
| Higher humidity correlates with higher PM2.5 in winter conditions | Query 5 |
| Hazardous-air days are concentrated in Nov-Jan | Query 6 |

## What I did (for the class presentation)

> "I handled the analytics layer of our pipeline. I wrote six SQL queries
> against the normalized SQLite database that Member 2 built. Each query
> uses JOINs across the stations, pollutants, and measurements tables to
> answer real questions about Delhi's air quality — monthly trends, the
> worst hours of the day, seasonal patterns, the relationship between
> humidity and pollution, and how often pollution reaches hazardous levels.
> I also wrote the project README and documentation."

## Key SQL concepts I used
- **JOIN** — combining rows from multiple tables based on a key
- **GROUP BY** — collapsing rows into groups and applying aggregates
- **CASE WHEN** — conditional buckets (used for season and humidity grouping)
- **Aggregates** (AVG, COUNT, MIN, MAX) — summarizing groups
- **WHERE** filtering — narrowing rows before aggregation


In [ ]:
conn.close()
print("Done!")